In [4]:
!pip install tensorflow 



[notice] A new release of pip is available: 23.0.1 -> 26.0.1
[notice] To update, run: python.exe -m pip install --upgrade pip


     ---------------------------------------- 0.0/350.6 MB ? eta -:--:--
     --------------------------------------- 0.9/350.6 MB 28.2 MB/s eta 0:00:13
     --------------------------------------- 2.9/350.6 MB 37.1 MB/s eta 0:00:10
      -------------------------------------- 5.6/350.6 MB 44.8 MB/s eta 0:00:08
      -------------------------------------- 8.2/350.6 MB 52.0 MB/s eta 0:00:07
     - ------------------------------------ 10.6/350.6 MB 50.4 MB/s eta 0:00:07
     - ------------------------------------ 13.0/350.6 MB 54.7 MB/s eta 0:00:07
     - ------------------------------------ 15.5/350.6 MB 50.4 MB/s eta 0:00:07
     -- ----------------------------------- 18.5/350.6 MB 54.4 MB/s eta 0:00:07
     -- ----------------------------------- 20.2/350.6 MB 50.4 MB/s eta 0:00:07
     -- ----------------------------------- 21.5/350.6 MB 46.7 MB/s eta 0:00:08
     -- ----------------------------------- 22.1/350.6 MB 38.6 MB/s eta 0:00:09
     -- ----------------------------------- 22.

In [1]:
# ===========================================
# 1) Imports and setup
# ===========================================
import os
import random
import numpy as np
import pandas as pd
import tensorflow as tf
from sklearn.model_selection import train_test_split
from sklearn.metrics import classification_report, confusion_matrix
import matplotlib.pyplot as plt
import seaborn as sns

SEED = 42
random.seed(SEED)
np.random.seed(SEED)
tf.random.set_seed(SEED)

print("TensorFlow version:", tf.__version__)




TensorFlow version: 2.21.0


In [7]:
# ===========================================
# 2) Dataset path
#    Expected structure:
#    DATASET_DIR/
#      class_1/*.jpg
#      class_2/*.jpg
#      ...
# ===========================================
from pathlib import Path

# Option 1: set env var CLOTHES_DATASET_DIR to your local dataset folder.
# Option 2: keep placeholder; code will try local auto-detect, then Kaggle download.
DATASET_DIR = os.environ.get("CLOTHES_DATASET_DIR", r"C:\path\to\clothes-dataset")

IMG_EXTS = {".jpg", ".jpeg", ".png", ".bmp", ".webp"}

def looks_like_class_root(path: str) -> bool:
    if not os.path.isdir(path):
        return False
    class_dirs = [
        os.path.join(path, d)
        for d in os.listdir(path)
        if os.path.isdir(os.path.join(path, d))
    ]
    if len(class_dirs) < 2:
        return False
    # Class root should contain subfolders, each with image files.
    return any(
        any(os.path.splitext(f)[1].lower() in IMG_EXTS for f in os.listdir(cd))
        for cd in class_dirs
    )

def resolve_class_root(base_path: str) -> str | None:
    if not os.path.isdir(base_path):
        return None
    if looks_like_class_root(base_path):
        return base_path
    # Search one level deeper for folders like 'Clothes_Dataset'.
    for child in os.listdir(base_path):
        child_path = os.path.join(base_path, child)
        if looks_like_class_root(child_path):
            return child_path
    return None

candidate_dirs = [
    DATASET_DIR,
    str(Path.cwd() / "clothes-dataset"),
    str(Path.cwd() / "dataset" / "clothes-dataset"),
    str(Path.cwd() / "archive"),
]

resolved_dir = None
for candidate in candidate_dirs:
    resolved_dir = resolve_class_root(candidate)
    if resolved_dir is not None:
        break

if resolved_dir is None:
    try:
        import kagglehub
        downloaded_dir = kagglehub.dataset_download("ryanbadai/clothes-dataset")
        print("Downloaded dataset to:", downloaded_dir)
        resolved_dir = resolve_class_root(downloaded_dir)
    except Exception as e:
        raise FileNotFoundError(
            "Could not locate/download dataset folder. Set CLOTHES_DATASET_DIR to the class-root directory."
        ) from e

if resolved_dir is None:
    raise FileNotFoundError(
        "Dataset found but class-root folder not detected. Point CLOTHES_DATASET_DIR to the folder containing class subfolders."
    )

DATASET_DIR = resolved_dir
print("Using DATASET_DIR:", DATASET_DIR)

IMG_SIZE = (224, 224)
BATCH_SIZE = 32
AUTOTUNE = tf.data.AUTOTUNE
EPOCHS = 20


Downloaded dataset to: C:\Users\aadis\.cache\kagglehub\datasets\ryanbadai\clothes-dataset\versions\1
Using DATASET_DIR: C:\Users\aadis\.cache\kagglehub\datasets\ryanbadai\clothes-dataset\versions\1\Clothes_Dataset


In [8]:
# ===========================================
# 3) Build file list + labels
# ===========================================
allowed_ext = {".jpg", ".jpeg", ".png", ".bmp", ".webp"}
class_names = sorted(
    [d for d in os.listdir(DATASET_DIR) if os.path.isdir(os.path.join(DATASET_DIR, d))]
)
class_to_idx = {c: i for i, c in enumerate(class_names)}

rows = []
for cls in class_names:
    class_dir = os.path.join(DATASET_DIR, cls)
    for fname in os.listdir(class_dir):
        ext = os.path.splitext(fname)[1].lower()
        if ext in allowed_ext:
            rows.append((os.path.join(class_dir, fname), cls, class_to_idx[cls]))

df = pd.DataFrame(rows, columns=["filepath", "label_name", "label"])
print("Total images:", len(df))
print("Classes:", len(class_names))
display(df["label_name"].value_counts())



Total images: 7500
Classes: 15


label_name
Blazer            500
Celana_Panjang    500
Celana_Pendek     500
Gaun              500
Hoodie            500
Jaket             500
Jaket_Denim       500
Jaket_Olahraga    500
Jeans             500
Kaos              500
Kemeja            500
Mantel            500
Polo              500
Rok               500
Sweter            500
Name: count, dtype: int64

In [9]:
# ===========================================
# 4) Stratified split: 80 / 10 / 10
# ===========================================
train_df, temp_df = train_test_split(
    df, test_size=0.20, random_state=SEED, stratify=df["label"]
)
val_df, test_df = train_test_split(
    temp_df, test_size=0.50, random_state=SEED, stratify=temp_df["label"]
)

print("Train size:", len(train_df))
print("Val size:", len(val_df))
print("Test size:", len(test_df))

# Verify class distributions
print("\nTrain distribution:")
display(train_df["label_name"].value_counts(normalize=True).sort_index())
print("\nVal distribution:")
display(val_df["label_name"].value_counts(normalize=True).sort_index())
print("\nTest distribution:")
display(test_df["label_name"].value_counts(normalize=True).sort_index())



Train size: 6000
Val size: 750
Test size: 750

Train distribution:


label_name
Blazer            0.066667
Celana_Panjang    0.066667
Celana_Pendek     0.066667
Gaun              0.066667
Hoodie            0.066667
Jaket             0.066667
Jaket_Denim       0.066667
Jaket_Olahraga    0.066667
Jeans             0.066667
Kaos              0.066667
Kemeja            0.066667
Mantel            0.066667
Polo              0.066667
Rok               0.066667
Sweter            0.066667
Name: proportion, dtype: float64


Val distribution:


label_name
Blazer            0.066667
Celana_Panjang    0.066667
Celana_Pendek     0.066667
Gaun              0.066667
Hoodie            0.066667
Jaket             0.066667
Jaket_Denim       0.066667
Jaket_Olahraga    0.066667
Jeans             0.066667
Kaos              0.066667
Kemeja            0.066667
Mantel            0.066667
Polo              0.066667
Rok               0.066667
Sweter            0.066667
Name: proportion, dtype: float64


Test distribution:


label_name
Blazer            0.066667
Celana_Panjang    0.066667
Celana_Pendek     0.066667
Gaun              0.066667
Hoodie            0.066667
Jaket             0.066667
Jaket_Denim       0.066667
Jaket_Olahraga    0.066667
Jeans             0.066667
Kaos              0.066667
Kemeja            0.066667
Mantel            0.066667
Polo              0.066667
Rok               0.066667
Sweter            0.066667
Name: proportion, dtype: float64

In [ ]:
# ===========================================
# 5) tf.data pipeline helpers
# ===========================================
def decode_and_resize(path, label):
    img = tf.io.read_file(path)
    img = tf.io.decode_image(img, channels=3, expand_animations=False)
    img = tf.image.resize(img, IMG_SIZE)
    img = tf.cast(img, tf.float32) / 255.0
    return img, label

def make_dataset(dataframe, shuffle=False):
    paths = dataframe["filepath"].values
    labels = dataframe["label"].values.astype(np.int32)
    ds = tf.data.Dataset.from_tensor_slices((paths, labels))
    ds = ds.map(decode_and_resize, num_parallel_calls=AUTOTUNE)
    if shuffle:
        ds = ds.shuffle(buffer_size=len(dataframe), seed=SEED, reshuffle_each_iteration=True)
    ds = ds.batch(BATCH_SIZE).prefetch(AUTOTUNE)
    return ds

train_ds_clean = make_dataset(train_df, shuffle=True)
val_ds_clean = make_dataset(val_df, shuffle=False)
test_ds_clean = make_dataset(test_df, shuffle=False)



In [ ]:
# ===========================================
# 6) Augmentation blocks
#    Baseline model: NO augmentation
#    Augmented model:
#      - Train: Translation + Rotation
#      - Val/Test robustness: Flipping + Cropping + Color changes
# ===========================================
train_aug_layer = tf.keras.Sequential([
    tf.keras.layers.RandomTranslation(height_factor=0.10, width_factor=0.10, seed=SEED),  # Translation
    tf.keras.layers.RandomRotation(factor=0.15, seed=SEED),                                # Rotation
], name="train_aug_translate_rotate")

def eval_robust_augment(images, labels):
    # Flipping
    images = tf.image.random_flip_left_right(images, seed=SEED)

    # Cropping (resize up then random crop back)
    up_h, up_w = 256, 256
    images = tf.image.resize(images, (up_h, up_w))
    images = tf.image.random_crop(images, size=[tf.shape(images)[0], IMG_SIZE[0], IMG_SIZE[1], 3], seed=SEED)

    # Changing colors
    images = tf.image.random_brightness(images, max_delta=0.20, seed=SEED)
    images = tf.image.random_contrast(images, lower=0.8, upper=1.2, seed=SEED)
    images = tf.image.random_saturation(images, lower=0.8, upper=1.2, seed=SEED)
    images = tf.image.random_hue(images, max_delta=0.02, seed=SEED)

    images = tf.clip_by_value(images, 0.0, 1.0)
    return images, labels

val_ds_robust = val_ds_clean.map(eval_robust_augment, num_parallel_calls=AUTOTUNE).prefetch(AUTOTUNE)
test_ds_robust = test_ds_clean.map(eval_robust_augment, num_parallel_calls=AUTOTUNE).prefetch(AUTOTUNE)



In [ ]:
# ===========================================
# 7) CNN model
# ===========================================
def build_cnn(num_classes, use_train_aug=False):
    inputs = tf.keras.Input(shape=(IMG_SIZE[0], IMG_SIZE[1], 3))
    x = inputs
    if use_train_aug:
        x = train_aug_layer(x)

    x = tf.keras.layers.Conv2D(32, 3, padding="same", activation="relu")(x)
    x = tf.keras.layers.MaxPooling2D()(x)
    x = tf.keras.layers.Conv2D(64, 3, padding="same", activation="relu")(x)
    x = tf.keras.layers.MaxPooling2D()(x)
    x = tf.keras.layers.Conv2D(128, 3, padding="same", activation="relu")(x)
    x = tf.keras.layers.MaxPooling2D()(x)
    x = tf.keras.layers.Conv2D(256, 3, padding="same", activation="relu")(x)
    x = tf.keras.layers.GlobalAveragePooling2D()(x)
    x = tf.keras.layers.Dropout(0.3)(x)
    outputs = tf.keras.layers.Dense(num_classes, activation="softmax")(x)

    model = tf.keras.Model(inputs, outputs)
    model.compile(
        optimizer=tf.keras.optimizers.Adam(1e-3),
        loss="sparse_categorical_crossentropy",
        metrics=["accuracy"]
    )
    return model

callbacks = [
    tf.keras.callbacks.EarlyStopping(
        monitor="val_accuracy", patience=5, mode="max", restore_best_weights=True
    ),
    tf.keras.callbacks.ReduceLROnPlateau(
        monitor="val_loss", factor=0.5, patience=2, min_lr=1e-6
    )
]



In [ ]:
# ===========================================
# 8) Baseline training (NO augmentation)
# ===========================================
baseline_model = build_cnn(num_classes=len(class_names), use_train_aug=False)
history_baseline = baseline_model.fit(
    train_ds_clean,
    validation_data=val_ds_clean,
    epochs=EPOCHS,
    callbacks=callbacks,
    verbose=1
)



In [ ]:
# ===========================================
# 9) Augmented training
#    Train uses translation+rotation
# ===========================================
aug_model = build_cnn(num_classes=len(class_names), use_train_aug=True)
history_aug = aug_model.fit(
    train_ds_clean,          # augmentation is inside model
    validation_data=val_ds_clean,
    epochs=EPOCHS,
    callbacks=callbacks,
    verbose=1
)



In [ ]:
# ===========================================
# 10) Evaluation: clean and robust-shifted sets
# ===========================================
def evaluate_model(name, model, ds, split_name):
    loss, acc = model.evaluate(ds, verbose=0)
    return {"model": name, "split": split_name, "loss": loss, "accuracy": acc}

results = []
results.append(evaluate_model("Baseline", baseline_model, val_ds_clean, "Val-Clean"))
results.append(evaluate_model("Baseline", baseline_model, test_ds_clean, "Test-Clean"))
results.append(evaluate_model("Baseline", baseline_model, val_ds_robust, "Val-RobustAug"))
results.append(evaluate_model("Baseline", baseline_model, test_ds_robust, "Test-RobustAug"))

results.append(evaluate_model("Augmented", aug_model, val_ds_clean, "Val-Clean"))
results.append(evaluate_model("Augmented", aug_model, test_ds_clean, "Test-Clean"))
results.append(evaluate_model("Augmented", aug_model, val_ds_robust, "Val-RobustAug"))
results.append(evaluate_model("Augmented", aug_model, test_ds_robust, "Test-RobustAug"))

results_df = pd.DataFrame(results).sort_values(["split", "model"]).reset_index(drop=True)
display(results_df)



In [ ]:
# ===========================================
# 11) Detailed report on test sets
# ===========================================
def get_preds_and_true(model, ds):
    y_true = []
    y_pred = []
    for x_batch, y_batch in ds:
        probs = model.predict(x_batch, verbose=0)
        preds = np.argmax(probs, axis=1)
        y_true.extend(y_batch.numpy().tolist())
        y_pred.extend(preds.tolist())
    return np.array(y_true), np.array(y_pred)

for model_name, model in [("Baseline", baseline_model), ("Augmented", aug_model)]:
    for split_name, ds in [("Test-Clean", test_ds_clean), ("Test-RobustAug", test_ds_robust)]:
        y_true, y_pred = get_preds_and_true(model, ds)
        print(f"\n=== {model_name} on {split_name} ===")
        print(classification_report(y_true, y_pred, target_names=class_names, digits=4))



In [ ]:
# ===========================================
# 12) Confusion matrix utility
# ===========================================
def plot_confusion(y_true, y_pred, labels, title):
    cm = confusion_matrix(y_true, y_pred)
    plt.figure(figsize=(8, 6))
    sns.heatmap(cm, annot=False, cmap="Blues", xticklabels=labels, yticklabels=labels)
    plt.title(title)
    plt.xlabel("Predicted")
    plt.ylabel("True")
    plt.tight_layout()
    plt.show()

# Example plot: Augmented model on robust test
y_true_aug_rob, y_pred_aug_rob = get_preds_and_true(aug_model, test_ds_robust)
plot_confusion(y_true_aug_rob, y_pred_aug_rob, class_names, "Augmented Model - Test Robust Augmented")

